In [29]:
import os
import json
import random
from openai import OpenAI
import pandas as pd
from tqdm import tqdm


# Using LLMs to get prediction answer

In [30]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Paths
base_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/pororo"
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")

# Load dataset
with open(qa_json_path, "r") as f:
    qa_data = json.load(f)["PororoQA"]

# Load descriptions
descriptions = pd.read_csv(description_csv_path)

def get_prediction(question, gif_paths, description, subtitles):
    images = []
    for gif_path in gif_paths:
        with open(gif_path, "rb") as gif_file:
            images.append(gif_file.read())

    prompt = f"""
    Question: {question}
    Description: {description}
    Subtitles: {subtitles}
    Images: {len(images)} GIF frames

    Provide an answer based on the description, subtitles, and images.
    """

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150,
        temperature=0.7,
    )
    return completion.choices[0].message.content.strip()

# Check predicted answer

In [ ]:
# def check_answer(correct_answer, predicted_answer):
#     prompt = f"""
#     Correct Answer: {correct_answer}
#     Predicted Answer: {predicted_answer}

#     Is the predicted answer correct? Answer with "1" for correct and "0" for incorrect.
#     """

#     completion = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}],
#         max_tokens=10,
#         temperature=0.3,
#     )
#     return int(completion.choices[0].message.content.strip())

def check_answer(correct_answer, predicted_answer):
    """
    Compare the correct answer with the predicted answer using semantic similarity
    Returns: float between 0 and 1 indicating the similarity/correctness
    """
    prompt = f"""
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
    - 1.0: Perfect match or completely correct meaning
    - 0.75: Mostly correct with minor differences
    - 0.5: Partially correct
    - 0.25: Slightly correct but missing key points
    - 0.0: Completely incorrect or unrelated

    Provide only the numeric score (e.g. 0.75) with no other text.
    """

    completion = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0.3  # Lower temperature for more consistent scoring
    )
    
    try:
        score = float(completion.choices[0].message.content.strip())
        # Ensure score is between 0 and 1
        return max(0.0, min(1.0, score))
    except ValueError:
        # Return 0 if we can't parse the score
        return 0.0


# Processing Data and Calculating Accuracy

In [32]:
# Set random seed for reproducibility
random.seed(42)

# First, filter questions only for episode 1
ep1_questions = [q for q in qa_data if "Pororo_ENGLISH1_1_ep1" in q["video_name"]]

# Group questions by supporting_num for episode 1
grouped_questions = {}
for entry in ep1_questions:
    supporting_num = entry["supporting_num"]
    if supporting_num not in grouped_questions:
        grouped_questions[supporting_num] = []
    grouped_questions[supporting_num].append(entry)

# Ensure we process GIFs 1-36 in sequence
gif_numbers = list(range(1, 37))
correct_count = 0
total_count = len(gif_numbers)
gif_accuracy = {}

# Process each GIF number in sequence
for gif_num in tqdm(gif_numbers, total=total_count):
    # Get questions for current GIF number
    current_gif_questions = grouped_questions.get(str(gif_num), [])
    
    if not current_gif_questions:
        print(f"No questions found for GIF {gif_num}")
        continue
        
    # Select one random question for this GIF
    entry = random.choice(current_gif_questions)
    
    video_name = entry["video_name"]
    question = entry["question"]
    correct_idx = entry["correct_idx"]
    answers = [entry[f"answer{i}"] for i in range(5)]
    correct_answer = answers[correct_idx]
    qid = entry["qid"]

    # Construct paths
    episode_folder = os.path.join(base_dir, "Scenes_Dialogues", "Pororo_ENGLISH1_1", "Pororo_ENGLISH1_1_ep1")
    subtitles_path = os.path.join(episode_folder, "subtitles.txt")
    
    # Load subtitles
    with open(subtitles_path, "r") as f:
        subtitles = f.read()

    # Process current GIF
    gif_files = [f"{gif_num}.gif"]
    gif_paths = [os.path.join(episode_folder, gif_file) for gif_file in gif_files]

    # Get description
    description_row = descriptions.loc[descriptions.iloc[:, 0] == video_name]
    if description_row.empty:
        print(f"Description for {video_name} not found")
        continue
    description = description_row.iloc[0, 2]

    predicted_answer = get_prediction(question, gif_paths, description, subtitles)

    # Debugging prints
    print(f"GIF number: {gif_num}")
    print(f"QID: {qid}")
    print(f"Question: {question}")
    print(f"Correct Answer: {correct_answer}")
    print(f"Predicted Answer: {predicted_answer}")

    is_correct = predicted_answer is not None and check_answer(correct_answer, predicted_answer)
    correct_count += is_correct

    # Update gif accuracy
    if gif_num not in gif_accuracy:
        gif_accuracy[gif_num] = {"correct": 0, "total": 0}
    gif_accuracy[gif_num]["total"] += 1
    gif_accuracy[gif_num]["correct"] += is_correct

    # Print accuracy for current GIF
    accuracy = gif_accuracy[gif_num]["correct"] / gif_accuracy[gif_num]["total"]
    print(f"GIF number: {gif_num}, Accuracy: {accuracy * 100:.0f}%")

# Calculate and print overall accuracy
average_accuracy = correct_count / total_count
print(f"Average Accuracy: {average_accuracy * 100:.2f}%")

  0%|          | 0/36 [00:00<?, ?it/s]

GIF number: 1
QID: 14
Question: what is the characteristic of village
Correct Answer: village is covered by snow and surrounded by forest
Predicted Answer: The characteristic of the village described is that it is located in a cold country, surrounded by a forest with many trees covered in snow. In this village lives a curious little penguin named Pororo, who embarks on adventures, including discovering a baby dinosaur. The village appears to be a close-knit and whimsical place, where friendships are formed, as seen in Pororo's interaction with the baby dinosaur, Crong. The snowy environment and the presence of unique characters highlight the village's charm and the playful nature of its inhabitants.


  3%|▎         | 1/36 [00:03<02:11,  3.75s/it]

GIF number: 1, Accuracy: 50%
GIF number: 2
QID: 58
Question: what kind of penguin is pororo
Correct Answer: pororo is a curious little penguin
Predicted Answer: Pororo is a little penguin who lives in a snowy, cold country. He is portrayed as curious and adventurous. In the story, he finds an egg that hatches into a baby dinosaur, which he initially thinks is a scary monster but eventually befriends. The setting is a small village in a snowy forest, highlighting the chilly environment where Pororo lives.


  6%|▌         | 2/36 [00:07<02:09,  3.80s/it]

...

GIF number: 34, Accuracy: 25%
GIF number: 35
QID: 1119
Question: was pororo enjoying the new ride?
Correct Answer: yes, pororo was enjoying and said " yahoo"
Predicted Answer: Based on the provided description and subtitles, there is no indication that Pororo is enjoying a new ride. The content primarily focuses on Pororo's interaction with a baby dinosaur named Crong and their introduction as friends. There are no mentions of a ride or any related enjoyment in the text.


 97%|█████████▋| 35/36 [01:22<00:02,  2.38s/it]

GIF number: 35, Accuracy: 0%
GIF number: 36
QID: 674
Question: who did pororo make friends with
Correct Answer: pororo made friends with the baby dinosaur crong
Predicted Answer: Pororo made friends with a baby dinosaur named Crong.


100%|██████████| 36/36 [01:24<00:00,  2.34s/it]

GIF number: 36, Accuracy: 100%
Average Accuracy: 41.67%
